In [ ]:
# ─── 1. Install ───────────────────────────────────────────────────────────────
!pip uninstall -y peft accelerate transformers -q
!pip install -q \
  transformers==4.36.2 \
  accelerate==0.25.0 \
  datasets==2.16.1 \
  safetensors==0.4.2 \
  scikit-learn \
  numpy==1.26.4


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.8/126.8 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.7/265.7 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.1/507.1 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.4/166.4 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 72.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 kB 8.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not curr

In [ ]:
# ─── 2. Mount Drive ───────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/BERT_IFND_Final"
import os, json, time
os.makedirs(SAVE_DIR, exist_ok=True)

# ─── 3. Imports ───────────────────────────────────────────────────────────────
import torch
import numpy as np
import pandas as pd
import functools
from torch import nn
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from transformers import BertTokenizer, BertForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from datasets import load_dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# ─── 4. Load dataset ──────────────────────────────────────────────────────────
print("⏳ Loading dataset IFND-multimodal...")
hf_dataset = load_dataset("Nhat243/IFND-multimodal")
print(hf_dataset)

# Kiểm tra label distribution
train_labels_list = hf_dataset['train']['label']
print(f"📊 Label distribution - Train: REAL={list(train_labels_list).count(1)}, FAKE={list(train_labels_list).count(0)}")

# ─── 5. Preprocess với dataset.map() ─────────────────────────────────────────
model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)

def preprocess(example):
    inputs = tokenizer(
        str(example['text']),
        truncation=True,
        padding="max_length",
        max_length=128,
        return_tensors="pt"
    )
    return {
        "input_ids": inputs["input_ids"][0],
        "attention_mask": inputs["attention_mask"][0],
        "labels": int(example["label"])  # IFND: 1=REAL, 0=FAKE
    }

print("⏳ Preprocessing dataset...")
encoded_dataset = hf_dataset.map(
    preprocess,
    remove_columns=hf_dataset["train"].column_names,
    desc="Preprocessing"
)
encoded_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

train_dataset = encoded_dataset["train"]
val_dataset = encoded_dataset["validation"]
test_dataset = encoded_dataset["test"]

print(f"✅ Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

# ─── 6. Model ─────────────────────────────────────────────────────────────────
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n📊 BERT Model on IFND")
print(f"   Total params    : {total_params/1e6:.2f}M")
print(f"   Trainable params: {trainable_params/1e6:.2f}M")
print(f"   Alignment: REAL=1, FAKE=0")

# ─── 7. Metrics ───────────────────────────────────────────────────────────────
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds = np.argmax(probs, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary")
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": roc_auc_score(labels, probs[:, 1])
    }

# ─── 8. Training Args (LR = 1e-5 để đồng nhất) ───────────────────────────────
training_args = TrainingArguments(
    output_dir=SAVE_DIR,
    overwrite_output_dir=True,
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=1e-5,  # 🔥 SỬA: 2e-5 → 1e-5 (đồng nhất với các method khác)
    weight_decay=0.01,
    warmup_ratio=0.1,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=0,
    report_to="none",
    logging_steps=50
)

# ─── 9. Trainer ───────────────────────────────────────────────────────────────
torch.load = functools.partial(torch.load, weights_only=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# ─── 10. Training ─────────────────────────────────────────────────────────────
print("\n🚀 Training BERT on IFND (LR=1e-5)...")
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

start_train = time.time()
trainer.train()
train_time_min = (time.time() - start_train) / 60
peak_vram = torch.cuda.max_memory_allocated() / (1024**3) if torch.cuda.is_available() else 0

# Lưu model
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"✅ Model saved to: {SAVE_DIR}")
print(f"⏱️ Training time: {train_time_min:.2f} minutes")
print(f"💾 Peak VRAM: {peak_vram:.2f} GB")

# ─── 11. Latency Measurement ──────────────────────────────────────────────────
model.eval()
dummy_text = "This is a sample news headline for inference measurement on IFND dataset."
dummy_inputs = tokenizer(
    dummy_text,
    truncation=True,
    padding="max_length",
    max_length=128,
    return_tensors="pt"
)
input_ids = dummy_inputs["input_ids"].to(device)
attention_mask = dummy_inputs["attention_mask"].to(device)

# Warmup
with torch.no_grad():
    for _ in range(50):
        _ = model(input_ids=input_ids, attention_mask=attention_mask)

# Measure latency
latencies = []
with torch.no_grad():
    for _ in range(200):
        if device == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        _ = model(input_ids=input_ids, attention_mask=attention_mask)
        if device == "cuda":
            torch.cuda.synchronize()
        latencies.append((time.perf_counter() - t0) * 1000)

latencies = np.array(latencies)
latency_mean = np.mean(latencies)
latency_p50 = np.percentile(latencies, 50)
latency_p95 = np.percentile(latencies, 95)

print(f"\n⚡ Inference Latency (batch_size=1):")
print(f"   Mean: {latency_mean:.2f} ms")
print(f"   P50 : {latency_p50:.2f} ms")
print(f"   P95 : {latency_p95:.2f} ms")

# ─── 12. Test Evaluation ──────────────────────────────────────────────────────
print("\n📊 Evaluating on IFND TEST split...")
out = trainer.predict(test_dataset)

acc = out.metrics.get("test_accuracy", 0)
precision = out.metrics.get("test_precision", 0)
recall = out.metrics.get("test_recall", 0)
f1 = out.metrics.get("test_f1", 0)
auc = out.metrics.get("test_auc", 0)

# ─── 13. Results ─────────────────────────────────────────────────────────────
results = {
    "Dataset": "IFND-multimodal",
    "Model": "BERT-base-uncased",
    "Method": "Full Fine-Tuning",
    "Label_Mapping": "1=REAL, 0=FAKE",
    "Test_Results": {
        "Accuracy (%)": round(acc * 100, 2),
        "Precision (%)": round(precision * 100, 2),
        "Recall (%)": round(recall * 100, 2),
        "F1 (%)": round(f1 * 100, 2),
        "AUC": round(auc, 4)
    },
    "Latency_ms": {
        "Mean": round(latency_mean, 2),
        "P50": round(latency_p50, 2),
        "P95": round(latency_p95, 2)
    },
    "Hardware_Stats": {
        "Total_Params_M": round(total_params / 1e6, 2),
        "Trainable_Params_M": round(trainable_params / 1e6, 2),
        "Training_Time_Min": round(train_time_min, 2),
        "Peak_VRAM_GB": round(peak_vram, 2),
        "Learning_Rate": 1e-5,
        "Batch_Size": 16,
        "Epochs": 5,
        "Max_Length": 128
    }
}

# Hiển thị kết quả
print("\n" + "="*60)
print("📊 KẾT QUẢ BERT TRÊN IFND")
print("="*60)
print(f"📍 Test Set: IFND-multimodal")
print(f"   Accuracy : {results['Test_Results']['Accuracy (%)']}%")
print(f"   Precision: {results['Test_Results']['Precision (%)']}%")
print(f"   Recall   : {results['Test_Results']['Recall (%)']}%")
print(f"   F1 Score : {results['Test_Results']['F1 (%)']}%")
print(f"   AUC      : {results['Test_Results']['AUC']}")
print(f"\n⚡ Performance:")
print(f"   Latency (P50): {latency_p50:.2f} ms/sample")
print(f"   Training Time: {train_time_min:.2f} min")
print(f"   Peak VRAM    : {peak_vram:.2f} GB")
print("="*60)

# Lưu JSON
with open(os.path.join(SAVE_DIR, "results_IFND_BERT.json"), "w") as f:
    json.dump(results, f, indent=4)

# Lưu CSV
df_results = pd.DataFrame([results["Test_Results"]])
df_results.to_csv(os.path.join(SAVE_DIR, "results_IFND_BERT.csv"))

print(f"\n✅ Results saved to: {SAVE_DIR}")
print(f"   - results_IFND_BERT.json")
print(f"   - results_IFND_BERT.csv")
print(f"   - config.json and model files")

Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


Using device: cuda
⏳ Loading dataset IFND-multimodal...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/8416 [00:00<?, ? examples/s]

...


📊 BERT Model on IFND
   Total params    : 109.48M
   Trainable params: 109.48M
   Alignment: REAL=1, FAKE=0

🚀 Training BERT on IFND (LR=1e-5)...


/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:439: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Auc
1,0.054500,0.034641,0.991445,0.992583,0.996278,0.994427,0.998994
2,0.028900,0.042205,0.993346,0.996273,0.995037,0.995655,0.999178
3,0.011100,0.037907,0.993346,0.995043,0.996278,0.995660,0.997188
4,0.000500,0.041745,0.994297,0.996278,0.996278,0.996278,0.998898
5,0.000400,0.042846,0.993346,0.996273,0.995037,0.995655,0.998835


✅ Model saved to: /content/drive/MyDrive/BERT_IFND_Final
⏱️ Training time: 7.96 minutes
💾 Peak VRAM: 2.49 GB

⚡ Inference Latency (batch_size=1):
   Mean: 15.32 ms
   P50 : 12.71 ms
   P95 : 27.93 ms

📊 Evaluating on IFND TEST split...



📊 KẾT QUẢ BERT TRÊN IFND
📍 Test Set: IFND-multimodal
   Accuracy : 99.43%
   Precision: 99.38%
   Recall   : 99.88%
   F1 Score : 99.63%
   AUC      : 0.9995

⚡ Performance:
   Latency (P50): 12.71 ms/sample
   Training Time: 7.96 min
   Peak VRAM    : 2.49 GB

✅ Results saved to: /content/drive/MyDrive/BERT_IFND_Final
   - results_IFND_BERT.json
   - results_IFND_BERT.csv
   - config.json and model files


In [ ]:
print("⏳ Đang ngắt kết nối phiên làm việc để giải phóng GPU...")
from google.colab import runtime
time.sleep(10) # Đợi đồng bộ Drive
runtime.unassign()

⏳ Đang ngắt kết nối phiên làm việc để giải phóng GPU...
